## Agent 智能体

### 一、模型（Models）

#### 1.1 创建智能体

① 创建智能体，指定模型名，由Langchain初始化模型

In [6]:
from langchain.agents import create_agent
agent = create_agent(model="deepseek-v4-flash")

② 创建智能体，并使用创建好的model

In [5]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "enabled"}} # 指定思考模式 "enabled"、"disabled"
)
agent = create_agent(model=model)

#### 1.2 调用智能体

① 阻塞式调用，使用 `invoke` 方法：

In [66]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})

print(response['messages'][-1].content)

我是AI助手，由深度求索（DeepSeek）开发。我可以帮你回答问题、整理信息、写作、翻译、编程等。有什么我可以帮你的吗？


② 流式调用，只需要把调用方式改为 `stream`：

In [67]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？"}]},
    stream_mode="messages"
):
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

你好！我是 DeepSeek，由深度求索公司创造的 AI 助手。😊

我可以帮你解答问题、处理文件、进行对话交流等。我的一些特点包括：

- **免费使用**：目前完全免费，没有收费计划
- **多端支持**：可以通过 Web 端或 App 使用（支持语音输入）
- **文件处理**：支持上传图片、PDF、Word、Excel 等文件，读取其中的文字信息，也能识别图片中的可见内容
- **联网搜索**：支持联网查询（需要手动开启）
- **大上下文**：1M 上下文长度，可以处理很大的文本内容

有什么我可以帮你的吗？无论是学习、工作还是日常问题，都可以问我！✨

要注意，Agent 的 stream 模式同样返回一个 `generator`，但是其结构由 `stream_mode` 参数决定：
- `messages`: 返回 LLM 生成的每一个片段，是一个包含 `token` 和 `metadata` 的元组（Tuple）
- `updates`: 返回 Agent 运行过程中的每一次事件，例如与 LLM 的对话、工具的调用等
- `custom`: 返回通过 stream writer 记录的每一次自定义的输出


如果是为了流式输出AI返回的结果，使用 `messages` 模式即可。

### 二、消息（Messages）

In [70]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain.agents import create_agent

# 定义工具并调用
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    print('tool调用：get_weather')
    return f'{location} 城市， 今天是晴天！'

# 创建Agent
agent = create_agent(model="deepseek-v4-flash", tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage(content="你是智能问答助手"),
        HumanMessage(content="你好，我是虎哥"),
        AIMessage(content="你好，虎哥，很高兴认识你。"),
        HumanMessage(content="我的名字是什么？, 今天昆山天气如何？")
    ]
})

for message in response['messages']:
    message.pretty_print()

tool调用：get_weather
================================ System Message ================================

你是智能问答助手
================================ Human Message =================================

你好，我是虎哥
================================== Ai Message ==================================

你好，虎哥，很高兴认识你。
================================ Human Message =================================

我的名字是什么？, 今天昆山天气如何？
================================== Ai Message ==================================

虎哥，你刚才说你叫虎哥（虎哥是你的称呼）。我帮你查一下昆山的天气。
Tool Calls:
  get_weather (call_00_8Zzd1fh7NoamaBsqJvMu6390)
 Call ID: call_00_8Zzd1fh7NoamaBsqJvMu6390
  Args:
    location: 昆山
================================= Tool Message =================================
Name: get_weather

昆山 城市， 今天是晴天！
================================== Ai Message ==================================

虎哥，确认两件事：

1. **你的名字**：你自我介绍时说叫“虎哥”，所以我称你为虎哥。
2. **昆山天气**：今天是**晴天**。

需要我帮你做点别的吗，虎哥？


### 3. 提示词（Promts）

系统提示词（System Prompt）

In [53]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model = "deepseek-v4-flash",
    system_prompt="回答的时候要说好几次旺旺" # 设定角色、背景、任务等，每词调用都会跟用户问题一起拼接送给LLM
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

我是 AI 助手，由深度求索（DeepSeek）公司创造。可以帮你解答问题、处理信息、分析内容等。有什么需要我帮忙的吗？

提示词工程（Prompt Engineering）

In [52]:
# 提示词五要素
system_prompt = """
----------------提示词五要素-------------------
# 角色 Role：设定身份与专业视角、沟通风格和总体目标
# 任务 Task：明确要做什么，越具体越好
# 背景 Context：提供背景信息
# 约束 Constraints：不能做什么、要做到什么？比如字数、风格、禁用词、时间范围、是否允许编造等
# 输出格式 Format：规定输出格式，比如Markdown、JSON等

----------------Few-shot----------------
# 示例 Examples：输入输出样例
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-flash",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

我是 AI 助手，可以帮你完成写作、分析、翻译、编程、学习辅导等任务。你可以把具体需求告诉我，我会尽量清晰、准确地协助你。

结构化输出

In [72]:
import json
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.messages import HumanMessage

#首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

# 然后，我们就可以创建智能体并设置结构化输出的格式了。
agent = create_agent(
    model='deepseek-chat',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)

data_dict = response['structured_response'].model_dump()
pretty_json = json.dumps(data_dict, indent=4, ensure_ascii=False)
print(pretty_json)

{
    "name": "静海城",
    "location": "月球正面「静海」盆地西缘（北纬约17°，西经约34°），沿月海玄武岩平原与高地交界带呈带状展开，主体为地下穹顶与熔岩管改造的立体城区",
    "vibe": "寂静中的喧嚣。地表是灰白月壤与永久阴影，地下却是层层灯火——熔岩管里的「深街」终年暖光，穹顶花园用镜面导引地球反照；居民称地球为「抬头的那颗蓝」，每逢「地升节」全城熄灯仰望。空气里混着再生水的金属味与苔藓的土腥味，重力只有地球六分之一，走路像一场缓慢的舞。",
    "economy": "氦-3开采与同位素精炼为支柱产业，配套建设了月面最大的太阳能阵列与环形轨道电站；同时是地月贸易的中转枢纽与深空中继港，承担地球—火星航线的货运配载与船员轮换。衍生产业包括月壤3D打印建材、真空冶金、低重力医疗器械和「月球婚礼」观光业。"
}
